# 10.12 同樣幾張圖，怎樣知道它們發生在什麼時候？


一段四秒短片依序取出四張圖：圓、圓、方、方。若它們出現在 0、1000、2000、3000 毫秒，1500 毫秒時還是圓；若同樣四張圖改在 0、500、1000、1500 毫秒出現，此時已經是方。圖的順序相同，不能單靠「第幾張」回答「那個時刻看見什麼」。

影片中的一張圖叫**影格**（video frame）。`frame_ms` 是各影格開始生效的時間戳，單位毫秒（ms，1000 ms＝1 秒）。本例採 **sample-and-hold**，意思是下一格出現以前，畫面保持上一格；最後一格保持到片尾。這是本課短片的明示規則，不是從四張圖猜出它們之間所有動作。

圖上方是同一組四格；下方兩條色帶表示保持後的畫面。A 的圓持續到 2000 ms，B 只到 1000 ms；虛線查詢同一個 1500 ms，因此落在不同形狀的色帶上。


In [ ]:
# @title 準備本節的工具（首次執行）
from pathlib import Path
import os
import subprocess
import sys

# Colab 會先下載專案；本機 Jupyter 沿用已開啟的 repo。
try:
    import google.colab
except ImportError:
    in_colab = False
else:
    in_colab = True

if in_colab:
    root = Path("/content/tiny-perceptron-vlm")
    if not root.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/birdhackor/tiny-perceptron-vlm.git", str(root)],
            env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"},
            check=True,
        )
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-e", str(root)], check=True)
    os.chdir(root)

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("本機請先依 course/first-steps.md 開啟 repo；免安裝練習可使用本節的 Colab 入口")
sys.path.insert(0, str(root))
import torch

torch.set_num_threads(1)
torch.manual_seed(42)

In [ ]:
# @title 本節圖解
from IPython.display import SVG, display

display(SVG(filename=str(root / "course/figures/omni-10.12-frame-time.svg")))

下面直接畫 32×32 灰階圖，1 表示白、0 表示黑。`frames` 的軸是 `[T,C,H,W]`：影格數、通道、高、寬；這裡 `T=4`、灰階 `C=1`，沒有再加筆數軸。`at` 找出不晚於查詢時刻的最後一張影格。

`torch.arange(32)`先列出0到31，減15.5後讓座標中心落在圖片中央。`meshgrid(..., indexing="ij")`把它展成每個像素的兩張座標表：`y[i,j]=i-15.5`、`x[i,j]=j-15.5`，圓與方的不等式便可逐像素算出白黑值。四張圖經`stack`得到`[4,32,32]`；`[:, None]`保留全部影格，再插入長度1的灰階通道軸，才成為`[4,1,32,32]`。

程式用`torch.searchsorted(..., right=True)`計算查詢值應插在時間清單的哪個位置：放在所有小於或等於它的時間戳之後。A的1500應插在1000與2000之間，插入位置為2，減1就得到影格1；查2000時則放在2000之後，位置3減1得影格2。`right=True`因此讓切換時刻採用新的一格。


In [ ]:
import torch

axis = torch.arange(32) - 15.5
y, x = torch.meshgrid(axis, axis, indexing="ij")
circle = ((x.square() + y.square()) <= 8**2).float()
square = ((x.abs() <= 8) & (y.abs() <= 8)).float()
frames = torch.stack([circle, circle, square, square])[:, None]
times_a = torch.tensor([0.0, 1000.0, 2000.0, 3000.0])
times_b = torch.tensor([0.0, 500.0, 1000.0, 1500.0])
duration_ms = 4000.0


def validate(images, times, duration):
    if images.ndim != 4 or tuple(images.shape[1:]) != (1, 32, 32):
        raise ValueError("需要 [T,1,32,32] 灰階影格")
    if not 1 <= len(images) <= 16 or times.shape != (len(images),):
        raise ValueError("影格數須為 1–16，且每格有一個時間戳")
    if not torch.isfinite(images).all() or not torch.isfinite(times).all():
        raise ValueError("像素與時間須為有限數字")
    if not 0 < duration <= 4000:
        raise ValueError("片長須在 0–4000 ms 內")
    if times[0] != 0 or not (times[1:] > times[:-1]).all():
        raise ValueError("從 0 ms 有畫面，時間戳嚴格遞增")
    if not ((times >= 0) & (times < duration)).all():
        raise ValueError("影格不能在片外")


def at(images, times, duration, query_ms):
    validate(images, times, duration)
    if not 0 <= query_ms < duration:
        raise ValueError("查詢不能在片外")
    index = int(torch.searchsorted(times, torch.tensor(query_ms), right=True)) - 1
    return index, images[index]


for name, times in [("A", times_a), ("B", times_b)]:
    index, image = at(frames, times, duration_ms, 1500.0)
    print(name, "1500 ms 採影格", index)
print("A 在切換前／切換時的影格", [at(frames, times_a, duration_ms, t)[0] for t in [1999.0, 2000.0]])

# 這是待送入入口的資料記錄；pixels 仍是真正的像素，不是形狀答案。
records = []
for clip, times in [("A", times_a), ("B", times_b)]:
    validate(frames, times, duration_ms)
    records += [{"kind": "clip_begin", "clip": clip}, {"kind": "video_begin"}]
    records += [{"kind": "frame", "time_ms": float(t), "pixels": image} for t, image in zip(times, frames, strict=True)]
    records += [{"kind": "video_end"}, {"kind": "clip_end"}]
records += [{"kind": "text", "text": "A 在 1500 ms 是什麼形狀？"}]
print("記錄項數", len(records))
print("兩個片段的開始位置", [(i, row["clip"]) for i, row in enumerate(records) if row["kind"] == "clip_begin"])

前兩行印 A 採影格 `1`、B 採影格 `3`，對照最初的四張圖，就是圓與方；切換前／切換時印 `[1,2]`。這裡驗的是查時間的程式，沒有模型辨識形狀。片尾 4000 ms 已在片外，不能讓程式悄悄沿用最後一格作答。

後兩行印記錄項數 `17`、開始位置 `[(0,'A'),(8,'B')]`。`clip_begin`／`clip_end` 標出**片段界線**：A 與 B 各自從 0 ms 開始，不能混成同一條片內時間軸。`video_begin`／`video_end` 與最後的 `text` 標出**模態界線**：哪些是畫面、哪些是文字提問。標記本身不提供「圓」或「方」；圖片入口仍要處理 `pixels`。真正接進模型時，要把像素特徵與這些時間、界線一起表示，不能只保留一串圖的平均。本節先把材料定義清楚，還沒有選定位置編碼或訓練模型。

第19章助理（產品A）的新版影音輸入會容納最多16張這種灰階影格，共同時間軸最多四秒；本節的片段A／B各只有四張。練習只把 `times_b` 改成 `[0,500,1750,3000]`，先預測 B 在 1500 ms 採哪一格，再執行：應採影格 1，回到圓。再把第一個時間改成 100，確認程式拒絕，因為片頭到 100 ms 沒有已定義的畫面。

<details>
<summary>前文與後續</summary>

[10.2](https://birdhackor.github.io/tiny-perceptron-vlm/10.2.html)保留同一張圖內的圖塊順序，[10.4](https://birdhackor.github.io/tiny-perceptron-vlm/10.4.html)給圖塊位置提示；本節增加的是多張圖之間的時間與片段邊界。[12.17](https://birdhackor.github.io/tiny-perceptron-vlm/12.17.html)會把聲音放到同一條時間軸上。[10.7](https://birdhackor.github.io/tiny-perceptron-vlm/10.7.html)已示範把圖特徵換進文字序列；本節的 `records` 只是清楚展示輸入結構，沒有暗中呼叫另一個模型。

</details>
